In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping_update_calo.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_data.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_data_update_calo.df"
data_df = load_df(bnb_path, keys2load, 100)
data_pfp_df = data_df['pfp']
data_hit0_df = data_df['hit0']
data_hit1_df = data_df['hit1']
data_hit2_df = data_df['hit2']
data_hdr_df = data_df['hdr']

In [ ]:
mc_bnb_hit0_df.columns

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

if "pion" in bnb_path:
    data_tot_pot = 8.371e+19
else:
    data_tot_pot = 5.948e+18
    
print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))
data_pfp_df[pot_weight_col] = np.ones(len(data_pfp_df))


In [ ]:
import pandas as pd


def analyze_track_tuples(hit_dfs_dict, pfp_df, dataset_name="Dataset"):
    """Extracts unique 4-tuple track/slice identifiers from hit and PFP DataFrames

    and computes union and overlap statistics.
    """
    target_levels = [
        "__ntuple",
        "entry",
        "rec.slc..index",
        "rec.slc.reco.pfp..index",
    ]

    print(f"=== Running analysis for {dataset_name} ===")

    # 1. Combined Unique Combinations Across Hit DataFrames
    hit_tuple_sets = []
    total_hit_rows = 0

    for plane, df in hit_dfs_dict.items():
        if df is not None and not df.empty:
            total_hit_rows += len(df)
            # Extract target index levels directly using Index.droplevel or Index.get_level_values
            # converting to MultiIndex / Index tuples directly is significantly faster than to_frame()
            tuples_set = set(df.index.droplevel(
                [col for col in df.index.names if col not in target_levels]
            ).unique())
            hit_tuple_sets.append(tuples_set)

    combined_hit_tuples = set.union(*hit_tuple_sets) if hit_tuple_sets else set()
    n_unique_combined_hits = len(combined_hit_tuples)

    print(f"Total hit rows (plane 0+1+2): {total_hit_rows}")
    print(f"Unique track identifiers in hits: {n_unique_combined_hits}")

    # 2. Unique Combinations for PFP DataFrame
    if pfp_df is not None and not pfp_df.empty:
        # Get unique index tuples for target levels
        pfp_tuples = set(pfp_df.index.droplevel(
            [col for col in pfp_df.index.names if col not in target_levels]
        ).unique())
        n_unique_pfp = len(pfp_tuples)
        total_pfp_rows = len(pfp_df)
    else:
        pfp_tuples = set()
        n_unique_pfp = 0
        total_pfp_rows = 0

    print(f"Total PFP rows: {total_pfp_rows}")
    print(f"Unique track identifiers in PFP: {n_unique_pfp}")

    # 3. Overlap between Hits and PFP
    if combined_hit_tuples and pfp_tuples:
        overlap = len(combined_hit_tuples.intersection(pfp_tuples))
        print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

    print("\n" + "=" * 50 + "\n")

    return {
        "hit_tuples": combined_hit_tuples,
        "pfp_tuples": pfp_tuples,
    }


# --- Define Input Data Structures ---

mc_bnb_hit_dfs = {
    "0": mc_bnb_hit0_df,
    "1": mc_bnb_hit1_df,
    "2": mc_bnb_hit2_df,
}

data_hit_dfs = {
    "0": data_hit0_df,
    "1": data_hit1_df,
    "2": data_hit2_df,
}

# --- Execute Analysis ---

mc_results = analyze_track_tuples(
    hit_dfs_dict=mc_bnb_hit_dfs,
    pfp_df=mc_bnb_pfp_df,
    dataset_name="MC BNB",
)

data_results = analyze_track_tuples(
    hit_dfs_dict=data_hit_dfs,
    pfp_df=data_pfp_df,
    dataset_name="Data",
)

In [ ]:
import pandas as pd


def calculate_ptype_breakdown(
    df,
    p_type_col,
    weight_col=None,
    dataset_name="Dataset",
    p_type_labels=None,
):
    """Calculates and prints unweighted and weighted particle type (p_type)

    yields and percentages for a given PFP DataFrame.
    """
    print("\n" + "=" * 65)
    print(f" Particle Composition Breakdown: {dataset_name} ")
    print("=" * 65)

    if df is None or df.empty:
        print(f"Warning: {dataset_name} DataFrame is empty or None.")
        return None

    # Check if p_type column exists
    if p_type_col not in df.columns:
        print(
            f"Error: Column {p_type_col} not found in {dataset_name} DataFrame."
        )
        return None

    # 1. Filter out missing p_type values
    valid_mask = df[p_type_col].notna()
    df_clean = df[valid_mask].copy()

    if df_clean.empty:
        print(f"Warning: No valid non-null entries found for {p_type_col}.")
        return None

    # 2. Extract weights if available, else default to 1.0
    if weight_col and weight_col in df_clean.columns:
        weights = df_clean[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=df_clean.index)

    # 3. Calculate Weighted & Unweighted Statistics
    stats_df = pd.DataFrame(
        {"p_type": df_clean[p_type_col], "weight": weights}
    )

    summary = (
        stats_df.groupby("p_type")
        .agg(Counts=("weight", "count"), Weighted_Yield=("weight", "sum"))
        .reset_index()
    )

    total_counts = summary["Counts"].sum()
    total_weighted = summary["Weighted_Yield"].sum()

    summary["Raw_%"] = (summary["Counts"] / total_counts) * 100
    summary["Weighted_%"] = (
        (summary["Weighted_Yield"] / total_weighted) * 100
        if total_weighted > 0
        else 0.0
    )

    # Map readable category labels if provided (e.g. {13: 'muon', 211: 'pion'})
    if p_type_labels:
        summary["p_type_label"] = summary["p_type"].map(
            lambda x: p_type_labels.get(x, str(x))
        )
    else:
        summary["p_type_label"] = summary["p_type"].astype(str)

    # Sort by weighted yield descending
    summary = summary.sort_values(by="Weighted_Yield", ascending=False)

    # 4. Pretty Print Output
    header = f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}"
    print(header)
    print("-" * len(header))

    for _, row in summary.iterrows():
        label = row["p_type_label"]
        print(
            f"{label:<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | "
            f"{row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%"
        )

    print("-" * len(header))
    print(
        f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | "
        f"{total_weighted:<10.1f} | {100.0:<9.2f}%"
    )
    print("=" * 65 + "\n")

    return summary


# --- Configuration ---

p_type_col = ("pfp", "trk", "truth", "p", "p_type", "")
weight_col = ("slc", "wgt", "", "", "", "")

# Optional label mapping dictionary for clean printing
p_type_labels_map = {
    0: "Other",
    1: "shower",
    13: "μ",
    2212: "p",
    211: "inelastic pion",
    -211: "stopping pion",
}


# --- Execute on Both DataFrames ---

mc_bnb_summary = calculate_ptype_breakdown(
    df=mc_bnb_pfp_df,
    p_type_col=p_type_col,
    weight_col=weight_col,
    dataset_name="MC BNB",
    p_type_labels=p_type_labels_map,  # Pass dictionary or None
)

data_summary = calculate_ptype_breakdown(
    df=data_pfp_df,
    p_type_col=p_type_col,
    weight_col=None,  # Data typically has no event weights
    dataset_name="Data",
    p_type_labels=p_type_labels_map,
)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binned_statistic


def _compute_root_tprofile_stats(x_vals, y_vals, weights, bins_x):
    """Calculates ROOT-style TProfile statistics per bin:

    Weighted Mean <Y> and Standard Deviation sigma_Y.
    """
    bin_centers = 0.5 * (bins_x[:-1] + bins_x[1:])

    if len(x_vals) == 0:
        nan_arr = np.full_like(bin_centers, np.nan)
        return bin_centers, nan_arr, nan_arr, nan_arr

    # Binned sums
    sum_w_y, _, _ = binned_statistic(
        x_vals, y_vals * weights, statistic="sum", bins=bins_x
    )
    sum_w, _, _ = binned_statistic(
        x_vals, weights, statistic="sum", bins=bins_x
    )
    count, _, _ = binned_statistic(
        x_vals, weights, statistic="count", bins=bins_x
    )

    mean_y = np.full_like(sum_w, np.nan)
    valid_bins = sum_w > 0
    mean_y[valid_bins] = sum_w_y[valid_bins] / sum_w[valid_bins]

    # Weighted standard deviation (ROOT TProfile spread option 'S')
    sum_w_y2, _, _ = binned_statistic(
        x_vals, (y_vals**2) * weights, statistic="sum", bins=bins_x
    )

    std_y = np.full_like(sum_w, np.nan)
    std_err = np.full_like(sum_w, np.nan)

    variance = (sum_w_y2 / np.maximum(sum_w, 1e-9)) - (mean_y**2)
    variance = np.maximum(variance, 0.0)  # Numerical safety
    std_y[valid_bins] = np.sqrt(variance[valid_bins])

    # Standard Error on Mean: sigma / sqrt(N)
    std_err[valid_bins] = std_y[valid_bins] / np.sqrt(
        np.maximum(count[valid_bins], 1.0)
    )

    return bin_centers, mean_y, std_y, std_err


def _extract_clean_arrays(
    df, x_col, y_col, x_split_col, weight_col, y_max_cutoff=10.0
):
    """Safely filters non-null, finite values, and applies y_col <= y_max_cutoff mask."""
    if df is None or df.empty:
        return np.array([]), np.array([]), np.array([]), np.array([])

    mask = df[x_col].notna() & df[y_col].notna() & df[x_split_col].notna()
    plot_df = df[mask]

    x_vals = plot_df[x_col].values
    y_vals = plot_df[y_col].values
    split_vals = plot_df[x_split_col].values

    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).values
    else:
        weights = np.ones_like(x_vals)

    finite_and_cutoff_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(split_vals)
        & np.isfinite(weights)
        & (y_vals <= y_max_cutoff)
    )

    return (
        x_vals[finite_and_cutoff_mask],
        y_vals[finite_and_cutoff_mask],
        split_vals[finite_and_cutoff_mask],
        weights[finite_and_cutoff_mask],
    )


def plot_split_tpc_2d_with_root_profile(
    mc_df: pd.DataFrame,
    data_df: pd.DataFrame = None,
    x_col: str = "rr",
    y_col: str = "dedx",
    x_split_col: str = "x",
    mc_weight_col: str = None,
    data_weight_col: str = None,
    bins_x: np.ndarray = np.linspace(0, 150, 51),
    bins_y: np.ndarray = np.linspace(0, 10, 51),
    dedx_max_cutoff: float = 10.0,
    xlabel: str = "Residual Range [cm]",
    ylabel: str = "dE/dx [MeV/cm]",
    title_prefix: str = "Hit Distribution",
    cmap_name: str = "viridis",
    figsize: tuple = (16, 9),
):
    """Plots 2D histogram with ROOT-style TProfile in top plot

    and Data / MC ratio in the bottom plot.
    """
    mc_color = "#d62728"
    data_color = "#0f4c81"  # Dark navy blue

    # Styling settings for points and lines
    marker_size = 5.0
    line_width = 1.8
    cap_size = 3
    cap_thick = 1.5

    # 1. Extract Clean Data Arrays
    x_mc, y_mc, split_mc, w_mc = _extract_clean_arrays(
        mc_df, x_col, y_col, x_split_col, mc_weight_col, dedx_max_cutoff
    )
    x_data, y_data, split_data, w_data = _extract_clean_arrays(
        data_df, x_col, y_col, x_split_col, data_weight_col, dedx_max_cutoff
    )

    mc_neg, mc_pos = split_mc < 0, split_mc >= 0
    data_neg, data_pos = split_data < 0, split_data >= 0

    # 2. Setup Figure Layout with horizontal spacing (wspace=0.22)
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(
        2,
        3,
        height_ratios=[3, 1],
        width_ratios=[1, 1, 0.05],
        hspace=0.08,
        wspace=0.22,
    )

    ax_top_l = fig.add_subplot(gs[0, 0])
    ax_top_r = fig.add_subplot(gs[0, 1], sharey=ax_top_l)

    ax_bot_l = fig.add_subplot(gs[1, 0], sharex=ax_top_l)
    ax_bot_r = fig.add_subplot(gs[1, 1], sharex=ax_top_r, sharey=ax_bot_l)

    # Ensure right panel y-tick labels remain visible with wider separation
    plt.setp(ax_top_r.get_yticklabels(), visible=True)
    plt.setp(ax_bot_r.get_yticklabels(), visible=True)

    cax = fig.add_subplot(gs[0, 2])  # Colorbar axis

    cmap = globals().get("sunset_cmap", cmap_name)

    # Background Setup
    bg_x = x_mc if len(x_mc) > 0 else x_data
    bg_y = y_mc if len(y_mc) > 0 else y_data
    bg_split = split_mc if len(split_mc) > 0 else split_data
    bg_w = w_mc if len(w_mc) > 0 else w_data

    bg_neg, bg_pos = bg_split < 0, bg_split >= 0

    h_l, _, _ = np.histogram2d(
        bg_x[bg_neg], bg_y[bg_neg], bins=[bins_x, bins_y], weights=bg_w[bg_neg]
    )
    h_r, _, _ = np.histogram2d(
        bg_x[bg_pos], bg_y[bg_pos], bins=[bins_x, bins_y], weights=bg_w[bg_pos]
    )
    vmax = max(h_l.max(), h_r.max()) if max(h_l.max(), h_r.max()) > 0 else None

    # Top Plot 2D Background
    im0 = ax_top_l.hist2d(
        bg_x[bg_neg],
        bg_y[bg_neg],
        bins=[bins_x, bins_y],
        weights=bg_w[bg_neg],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]
    im1 = ax_top_r.hist2d(
        bg_x[bg_pos],
        bg_y[bg_pos],
        bins=[bins_x, bins_y],
        weights=bg_w[bg_pos],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    sides = [
        (ax_top_l, ax_bot_l, mc_neg, data_neg, "$X < 0$ cm"),
        (ax_top_r, ax_bot_r, mc_pos, data_pos, "$X \\geq 0$ cm"),
    ]

    for ax_top, ax_bot, mask_mc, mask_data, label in sides:
        # Calculate Stats (ROOT-style TProfile)
        cx, my_mc, std_mc, err_mc = _compute_root_tprofile_stats(
            x_mc[mask_mc], y_mc[mask_mc], w_mc[mask_mc], bins_x
        )
        _, my_data, std_data, err_data = _compute_root_tprofile_stats(
            x_data[mask_data], y_data[mask_data], w_data[mask_data], bins_x
        )

        # --- Top Plot: ROOT-style TProfile ---
        if len(x_mc[mask_mc]) > 0:
            ax_top.errorbar(
                cx,
                my_mc,
                yerr=std_mc,
                fmt="o",
                color=mc_color,
                ms=marker_size,
                elinewidth=line_width,
                capsize=cap_size,
                capthick=cap_thick,
                label="MC",
                zorder=10,
            )

        if len(x_data[mask_data]) > 0:
            ax_top.errorbar(
                cx,
                my_data,
                yerr=std_data,
                fmt="o",
                color=data_color,
                ms=marker_size,
                elinewidth=line_width,
                capsize=cap_size,
                capthick=cap_thick,
                label="Data",
                zorder=11,
            )

        # Top Formatting
        ax_top.set_title(f"{title_prefix}: {label}", fontsize=13, pad=8)
        ax_top.set_xlim(bins_x[0], bins_x[-1])
        ax_top.set_ylim(bins_y[0], bins_y[-1])
        ax_top.grid(alpha=0.3, linestyle="--")
        plt.setp(ax_top.get_xticklabels(), visible=False)

        # Top Legend with simplified labels ("MC" and "Data")
        ax_top.legend(
            loc="upper right",
            frameon=True,
            facecolor="white",
            framealpha=0.85,
            fontsize=15,
        )

        # --- Bottom Plot: Data / MC Ratio ---
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.divide(my_data, my_mc)
            ratio_err = np.divide(std_data, my_mc)  # Data_err / MC
            mc_rel_err = np.divide(std_mc, my_mc)   # Relative MC spread

        # 1. Draw MC Shaded Band around 1.0 representing relative MC spread
        valid_bins = np.isfinite(ratio) & np.isfinite(mc_rel_err)
        ax_bot.fill_between(
            cx,
            1.0 - mc_rel_err,
            1.0 + mc_rel_err,
            where=valid_bins,
            color=mc_color,
            alpha=0.25,
            zorder=8,
        )

        # 2. Draw Data / MC ratio points with Data_err / MC error bars
        ax_bot.errorbar(
            cx,
            ratio,
            yerr=ratio_err,
            fmt="o",
            color=data_color,
            ms=marker_size,
            elinewidth=line_width,
            capsize=cap_size,
            capthick=cap_thick,
            zorder=10,
        )
        ax_bot.axhline(1.0, color=mc_color, linestyle="--", linewidth=2.0)

        ax_bot.set_xlabel(xlabel, fontsize=12)
        ax_bot.grid(alpha=0.3, linestyle="--")

    # Y Labels
    ax_top_l.set_ylabel(ylabel, fontsize=12)
    ax_bot_l.set_ylabel("Data / MC", fontsize=11)

    # Colorbar
    cbar = fig.colorbar(im1, cax=cax)
    cbar.set_label("Weighted Entries", fontsize=11)

    # Hide extra bottom-right axis container
    ax_unused = fig.add_subplot(gs[1, 2])
    ax_unused.set_visible(False)

    return fig, (ax_top_l, ax_top_r, ax_bot_l, ax_bot_r)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define bin ranges for Residual Range and dE/dx
rr_bins = np.linspace(0, 80, 41)      # Residual Range [cm]
dedx_bins = np.linspace(0, 10, 51)     # dE/dx [MeV/cm]

# MultiIndex weight tuple (or string/None depending on your DF structure)
mc_weight = ('slc', 'wgt', '', '', '', '') 

# Loop through planes 0, 1, and 2
for plane_name in ['0', '1', '2']:
    mc_hit_df = mc_bnb_hit_dfs.get(plane_name)
    data_hit_df = data_hit_dfs.get(plane_name)

    fig, axes = plot_split_tpc_2d_with_root_profile(
        mc_df=mc_hit_df,
        data_df=data_hit_df,
        x_col="rr",
        y_col="dedx",
        x_split_col="x",
        mc_weight_col=mc_weight,
        data_weight_col=None,  # Data typically unweighted
        bins_x=rr_bins,
        bins_y=dedx_bins,
        xlabel="Residual Range [cm]",
        ylabel="Hit dE/dx [MeV/cm]",
        title_prefix=f"Plane {plane_name} Hit dE/dx vs RR",
    )
    

plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_dedx_by_rr_and_tpc(
    df: pd.DataFrame,
    data_df: pd.DataFrame = None,
    rr_range: tuple = (0.0, 1.0),
    dedx_col: str = "dedx",
    rr_col: str = "rr",
    x_col: str = "x",
    weight_col: str = None,
    data_weight_col: str = None,
    bins: np.ndarray = np.linspace(0.0, 10.0, 51),
    stacked: bool = False,
    density: bool = False,
    alpha: float = 0.3,
    linewidth: float = 1.8,
    marker_size: float = 5.0,
    title: str = None,
    ax_top: plt.Axes = None,
    ax_bot: plt.Axes = None,
):
    """Plots dE/dx distribution for MC (histograms) and Data (points) 

    with corresponding Data/MC ratios per X region (X > 0 and X <= 0).
    """

    # --- Helper: Apply identical filter and extract split arrays ---
    def _extract_and_split(data_frame, w_col):
        if data_frame is None or data_frame.empty:
            return None, None

        # Filter identically for both MC and Data
        mask = (
            data_frame[dedx_col].notna()
            & data_frame[rr_col].notna()
            & data_frame[x_col].notna()
            & (data_frame[rr_col] >= rr_range[0])
            & (data_frame[rr_col] < rr_range[1])
        )
        plot_df = data_frame[mask]
        if plot_df.empty:
            return None, None

        if w_col and w_col in plot_df.columns:
            w = plot_df[w_col].fillna(1.0)
        else:
            w = pd.Series(1.0, index=plot_df.index)

        # Categorization by X Position
        mask_pos = plot_df[x_col] > 0
        mask_neg = ~mask_pos

        grouped_data = [
            plot_df.loc[mask_pos, dedx_col],
            plot_df.loc[mask_neg, dedx_col],
        ]
        grouped_weights = [
            w[mask_pos],
            w[mask_neg],
        ]

        return grouped_data, grouped_weights

    # --- 1. Filter Data & MC Identically ---
    mc_data, mc_weights = _extract_and_split(df, weight_col)
    if mc_data is None:
        raise ValueError(
            f"No valid MC entries found for {rr_col} in range {rr_range}."
        )

    data_data, data_weights = _extract_and_split(data_df, data_weight_col)

    # --- 2. Identical Density Transformation (density=True) ---
    bin_width = bins[1] - bins[0]
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    def _apply_density_transform(grouped_w, is_stacked):
        if not density or grouped_w is None:
            return grouped_w

        if is_stacked:
            total_w = sum(w.sum() for w in grouped_w)
            if total_w > 0:
                scale = 1.0 / (total_w * bin_width)
                return [w * scale for w in grouped_w]
        else:
            normed = []
            for w in grouped_w:
                w_sum = w.sum()
                normed.append(
                    w / (w_sum * bin_width) if w_sum > 0 else w
                )
            return normed
        return grouped_w

    # Apply same density transform logic to both MC and Data
    mc_weights = _apply_density_transform(mc_weights, stacked)
    data_weights = _apply_density_transform(data_weights, False)

    colors = ["#1f77b4", "#d62728"]  # Blue ($X > 0$) & Red ($X \le 0$)
    labels = [r"MC: $X > 0$ cm", r"MC: $X \leq 0$ cm"]
    data_labels = [r"Data: $X > 0$ cm", r"Data: $X \leq 0$ cm"]

    # --- 3. Figure Layout Setup ---
    if ax_top is None or ax_bot is None:
        fig, (ax_top, ax_bot) = plt.subplots(
            2,
            1,
            figsize=(8, 6),
            sharex=True,
            gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
        )
    else:
        fig = ax_top.get_figure()

    # --- 4. Plot MC Histograms & Extract Individual MC Bins ---
    ax_top.hist(
        mc_data,
        bins=bins,
        weights=mc_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )

    ax_top.hist(
        mc_data,
        bins=bins,
        weights=mc_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # Compute unstacked individual MC bin counts for ratio division
    mc_unstacked_binned = []
    for idx in range(2):
        mc_c, _ = np.histogram(mc_data[idx], bins=bins, weights=mc_weights[idx])
        mc_unstacked_binned.append(mc_c)

    # --- 5. Plot Data Points & Calculate Specific Ratios ---
    if data_data is not None:
        for idx in range(2):  # idx 0: X > 0 (Blue), idx 1: X <= 0 (Red)
            d_vals = data_data[idx]
            d_w = data_weights[idx]

            # Compute Data binned values and errors
            data_c, _ = np.histogram(d_vals, bins=bins, weights=d_w)
            sum_w2, _ = np.histogram(d_vals, bins=bins, weights=d_w**2)
            data_err = np.sqrt(sum_w2)

            # Top plot: Data points
            valid_data = data_c > 0
            ax_top.errorbar(
                bin_centers[valid_data],
                data_c[valid_data],
                yerr=data_err[valid_data],
                fmt="o",
                color=colors[idx],
                ms=marker_size,
                elinewidth=linewidth,
                capsize=2,
                capthick=1.2,
                label=data_labels[idx],
                zorder=10 + idx,
            )

            # Bottom plot: Data / MC ratio per region
            # Data (X > 0) / MC (X > 0) AND Data (X <= 0) / MC (X <= 0)
            mc_c = mc_unstacked_binned[idx]

            with np.errstate(divide="ignore", invalid="ignore"):
                ratio = np.divide(data_c, mc_c)
                ratio_err = np.divide(data_err, mc_c)

            valid_ratio = np.isfinite(ratio) & (mc_c > 0)

            ax_bot.errorbar(
                bin_centers[valid_ratio],
                ratio[valid_ratio],
                yerr=ratio_err[valid_ratio],
                fmt="o",
                color=colors[idx],
                ms=marker_size,
                elinewidth=linewidth,
                capsize=2,
                capthick=1.2,
                zorder=10 + idx,
            )

    ax_bot.axhline(1.0, color="gray", linestyle="--", linewidth=1.5)

    # --- 6. Styling & Formatting ---
    ax_bot.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=13)

    if density:
        ax_top.set_ylabel("A.U.", fontsize=13)
    elif weight_col:
        ax_top.set_ylabel("Weighted Hits", fontsize=13)
    else:
        ax_top.set_ylabel("Hits", fontsize=13)

    ax_bot.set_ylabel("Data / MC", fontsize=11)

    if title is None:
        title = rf"Hit $dE/dx$ Distribution ({rr_range[0]} $\leq$ RR < {rr_range[1]} cm)"
    ax_top.set_title(title, fontsize=14, pad=10)

    ax_top.set_xlim(bins[0], bins[-1])
    ax_top.tick_params(axis="both", which="both", labelsize=11, direction="in")
    ax_bot.tick_params(axis="both", which="both", labelsize=11, direction="in")

    ax_top.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax_bot.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    ax_top.legend(
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )

    return fig, (ax_top, ax_bot)

In [ ]:
# Inputs
mc_hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
data_hit_dfs = [
    data_hit0_df,
    data_hit1_df,
    data_hit2_df,
]  # Optional: Set to [None, None, None] if no Data available
rr_ranges = [(3, 4), (6, 7), (15, 16)]

n_planes = len(mc_hit_dfs)

for rr_min, rr_max in rr_ranges:
    fig = plt.figure(figsize=(18, 7))
    gs = gridspec.GridSpec(
        2,
        n_planes,
        height_ratios=[3, 1],
        hspace=0.08,
        wspace=0.15,
    )

    top_axes = []
    bot_axes = []

    # Create top and bottom axes per plane
    for plane_idx in range(n_planes):
        ax_top = fig.add_subplot(gs[0, plane_idx])
        ax_bot = fig.add_subplot(
            gs[1, plane_idx], sharex=ax_top, sharey=bot_axes[0] if plane_idx > 0 else None
        )

        top_axes.append(ax_top)
        bot_axes.append(ax_bot)

    max_y_value = 0.0  # Track global maximum height for top plots

    for plane_idx, (mc_df_plane, data_df_plane) in enumerate(
        zip(mc_hit_dfs, data_hit_dfs)
    ):
        ax_t = top_axes[plane_idx]
        ax_b = bot_axes[plane_idx]

        try:
            plot_dedx_by_rr_and_tpc(
                df=mc_df_plane,
                data_df=data_df_plane,
                rr_range=(rr_min, rr_max),
                dedx_col="dedx",
                rr_col="rr",
                x_col="x",
                weight_col=None,
                bins=np.linspace(0.0, 10.0, 26),
                stacked=False,
                density=True,
                title=f"Plane {plane_idx}",
                ax_top=ax_t,
                ax_bot=ax_b,
            )

            # Record max y for top plot alignment
            current_max = ax_t.get_ylim()[1] / 1.15
            if current_max > max_y_value:
                max_y_value = current_max

        except ValueError:
            ax_t.set_title(f"Plane {plane_idx}: No Data", fontsize=14, pad=10)
            ax_b.set_visible(False)
            continue

        # Hide redundant y-axis labels and legends on non-leftmost panels
        if plane_idx > 0:
            ax_t.set_ylabel("")
            ax_b.set_ylabel("")
            plt.setp(ax_t.get_yticklabels(), visible=True)
            plt.setp(ax_b.get_yticklabels(), visible=True)

            legend = ax_t.get_legend()
            if legend:
                legend.remove()

        # Hide top x-axis tick labels (sharex with ratio)
        plt.setp(ax_t.get_xticklabels(), visible=False)

    # Synchronize Y-limits across all top plots
    if max_y_value > 0:
        for ax_t in top_axes:
            ax_t.set_ylim(0, max_y_value * 1.15)

    # Synchronize Y-limits for bottom ratio plots (0.0 to 2.0 default)
    for ax_b in bot_axes:
        ax_b.set_ylim(0.0, 2.0)

    fig.suptitle(
        rf"Normalized Hit $dE/dx$ Comparison ({rr_min} $\leq$ RR < {rr_max} cm)",
        fontsize=16,
        y=0.98,
    )

    plt.show()

In [ ]:
hfit = load_physics_classes()

In [ ]:

# ===========================================================================
# 7. Execution Driver Script
# ===========================================================================
pdg = 13
particle = "muon"
if "pion" in bnb_path:
    pdg = 211
    particle = "pion"

# 1. Theoretical MPV generator
theoretical_mpv = make_theoretical_mpv_func(hfit, pdg=pdg)

# 2. Analyze Data
data_results, data_params = analyze(
    data_hit_dfs,
    particle=particle,
    theoretical_mpv_func=theoretical_mpv,
    out_prefix="data_langau",
    make_summary_plots=False,
)

# 3. Analyze MC
mc_results, mc_params = analyze(
    mc_hit_dfs,
    particle=particle,
    theoretical_mpv_func=theoretical_mpv,
    out_prefix="mc_langau",
    make_summary_plots=False,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


import matplotlib.pyplot as plt
import numpy as np


def plot_all_planes(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle="muon",
    out_prefix="langau_rr",
    show_data_fits=True,
    show_mc_fits=True,
):
    # Always plot 3 columns: TPC 0, TPC 1, and Combined (-1)
    tpc_list = [0, 1, -1]
    tpc_titles = {
        0: r"$X < 0$ (TPC 0)",
        1: r"$X > 0$ (TPC 1)",
        -1: "Both TPCs Combined",
    }

    # Set figsize to (18, 6) matching 3:1 ratio for 3 square subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    for tpc_idx, tpc in enumerate(tpc_list):
        ax = axes[tpc_idx]

        for plane in range(3):
            color = PLANE_COLORS[plane]

            if show_data_fits:
                res_data = data_results.get((plane, tpc))
                popt_data, _ = data_params.get((plane, tpc), (None, None))
                if (
                    res_data is not None
                    and len(res_data)
                    and popt_data is not None
                ):
                    xs = np.linspace(
                        res_data["mpv_x"].min(), res_data["mpv_x"].max(), 200
                    )
                    label_data = f"Data P{plane}: {popt_data[0]:.2f} + {popt_data[1]:.2f}x^{popt_data[2]:.2f}"
                    ax.plot(
                        xs,
                        power_law(xs, *popt_data),
                        "-",
                        color=color,
                        linewidth=2.0,
                        label=label_data,
                    )

            if show_mc_fits:
                res_mc = mc_results.get((plane, tpc))
                popt_mc, _ = mc_params.get((plane, tpc), (None, None))
                if res_mc is not None and len(res_mc) and popt_mc is not None:
                    xs = np.linspace(
                        res_mc["mpv_x"].min(), res_mc["mpv_x"].max(), 200
                    )
                    label_mc = f"MC P{plane}: {popt_mc[0]:.2f} + {popt_mc[1]:.2f}x^{popt_mc[2]:.2f}"
                    ax.plot(
                        xs,
                        power_law(xs, *popt_mc),
                        "--",
                        color=color,
                        linewidth=2.0,
                        label=label_mc,
                    )

        ax.set_title(tpc_titles[tpc], fontsize=13)
        ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
        if tpc_idx == 0:
            ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

        ax.legend(fontsize=8, loc="upper left", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.6)

        # Enforce a 1:1 aspect ratio for the plot area (square axes)
        ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes_data_mc.pdf")
    plt.show()

import matplotlib.pyplot as plt
import numpy as np


def plot_by_plane(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle="muon",
    out_prefix="langau_rr",
    show_data_points=False,
    show_mc_points=False,
    show_data_fits=True,
    show_mc_fits=True,
):
    # Always plot 3 rows: TPC 1, TPC 0, and Combined (-1)
    tpcs = [1, 0, -1]
    row_labels = [
        r"$X > 0$ (TPC 1)",
        r"$X < 0$ (TPC 0)",
        "Both TPCs Combined",
    ]

    n_rows = len(tpcs)
    row_height = 5.0  # per-row plotting height
    fixed_overhead = (
        0.8  # inches reserved for titles, labels, and tick margins
    )
    fig_height = row_height * n_rows + fixed_overhead

    fig, axes = plt.subplots(
        n_rows, 3, figsize=(16, fig_height), sharex=True, sharey="row"
    )

    color_data = "#1f77b4"  # Blue
    color_mc = "#d62728"  # Red

    for row_idx, tpc in enumerate(tpcs):
        for plane in range(3):
            ax = axes[row_idx, plane]

            res_data = data_results.get((plane, tpc))
            popt_data, _ = data_params.get((plane, tpc), (None, None))

            if res_data is not None and len(res_data):
                if show_data_points:
                    ax.errorbar(
                        res_data["mpv_x"],
                        res_data["gsigma"],
                        yerr=res_data["gsigma_err"],
                        fmt="o",
                        color=color_data,
                        ms=4,
                        capsize=2,
                        label="Data Points",
                    )
                if show_data_fits and popt_data is not None:
                    xs = np.linspace(
                        res_data["mpv_x"].min(), res_data["mpv_x"].max(), 200
                    )
                    ax.plot(
                        xs,
                        power_law(xs, *popt_data),
                        "-",
                        color=color_data,
                        linewidth=1.8,
                        label=f"Data Fit: {popt_data[0]:.2f}+{popt_data[1]:.2f}x^{popt_data[2]:.2f}",
                    )

            res_mc = mc_results.get((plane, tpc))
            popt_mc, _ = mc_params.get((plane, tpc), (None, None))

            if res_mc is not None and len(res_mc):
                if show_mc_points:
                    ax.errorbar(
                        res_mc["mpv_x"],
                        res_mc["gsigma"],
                        yerr=res_mc["gsigma_err"],
                        fmt="s",
                        color=color_mc,
                        ms=3,
                        capsize=2,
                        label="MC Points",
                    )
                if show_mc_fits and popt_mc is not None:
                    xs = np.linspace(
                        res_mc["mpv_x"].min(), res_mc["mpv_x"].max(), 200
                    )
                    ax.plot(
                        xs,
                        power_law(xs, *popt_mc),
                        "--",
                        color=color_mc,
                        linewidth=1.8,
                        label=f"MC Fit: {popt_mc[0]:.2f}+{popt_mc[1]:.2f}x^{popt_mc[2]:.2f}",
                    )

            if row_idx == 0:
                ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
            if row_idx == n_rows - 1:
                ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
            if plane == 0:
                ax.set_ylabel(
                    f"{row_labels[row_idx]}\n" + r"$\sigma_G$ [MeV/cm]",
                    fontsize=12,
                )

            ax.legend(fontsize=7, loc="upper left", framealpha=0.9)
            ax.grid(True, linestyle=":", alpha=0.5)

            # Enforce 1:1 square aspect ratio for every subplot
            ax.set_box_aspect(1)

    fig.tight_layout(rect=[0, 0, 1, 1])
    fig.savefig(f"{out_prefix}_by_plane_data_mc.pdf")
    plt.show()

In [ ]:
# 4. Generate Data vs. MC comparison plots
plot_all_planes(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle=particle
)

plot_by_plane(
    data_results,
    data_params,
    mc_results,
    mc_params,
    particle=particle,
    out_prefix="comparison",
    show_data_points=True,  # Set to True
    show_mc_points=True,    # Set to True
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_data_vs_mc_fit(
    data_results,
    mc_results,
    mc_params,
    particle="muon",
    out_prefix="langau_rr",
    max_yerr=0.1,  # Added threshold for plotting
):
    # Always plot 3 rows: TPC 1, TPC 0, and Combined (-1)
    tpcs = [1, 0, -1]
    row_labels = [
        r"$X > 0$ (TPC 1)",
        r"$X < 0$ (TPC 0)",
        "Both TPCs Combined",
    ]

    n_rows = len(tpcs)
    row_height = 5.0  # per-row plotting height
    fixed_overhead = (
        0.8  # inches reserved for titles, labels, and tick margins
    )
    fig_height = row_height * n_rows + fixed_overhead

    fig, axes = plt.subplots(
        n_rows, 3, figsize=(16, fig_height), sharex=True, sharey="row"
    )

    color_data = "#1f77b4"  # Blue
    color_mc = "#d62728"  # Red

    for row_idx, tpc in enumerate(tpcs):
        for plane in range(3):
            ax = axes[row_idx, plane]

            raw_data = data_results.get((plane, tpc))
            res_mc = mc_results.get((plane, tpc))
            popt_mc, _ = mc_params.get((plane, tpc), (None, None))

            # Filter data points with gsigma_err <= max_yerr
            res_data = None
            if raw_data is not None and len(raw_data):
                mask = (
                    (raw_data["gsigma_err"] <= max_yerr)
                    & (raw_data["gsigma_err"] > 0)
                    & np.isfinite(raw_data["gsigma_err"])
                )
                res_data = raw_data[mask]

            # --- 1. Plot MC Fit Curve ---
            if popt_mc is not None and res_data is not None and len(res_data):
                x_min = res_data["mpv_x"].min()
                x_max = res_data["mpv_x"].max()
                xs = np.linspace(x_min, x_max, 200)

                label_mc = (
                    f"MC Fit: {popt_mc[0]:.2f} + {popt_mc[1]:.2f}x^{popt_mc[2]:.2f}"
                )
                ax.plot(
                    xs,
                    power_law(xs, *popt_mc),
                    "--",
                    color=color_mc,
                    linewidth=2.0,
                    label=label_mc,
                )

            # --- 2. Plot Filtered Data Points & Compute Chi2 / NDF ---
            if res_data is not None and len(res_data):
                ax.errorbar(
                    res_data["mpv_x"],
                    res_data["gsigma"],
                    yerr=res_data["gsigma_err"],
                    fmt="o",
                    color=color_data,
                    ms=4,
                    capsize=2,
                    label="Data Points",
                )

                if popt_mc is not None:
                    x_val = res_data["mpv_x"].to_numpy()
                    y_val = res_data["gsigma"].to_numpy()
                    yerr_val = res_data["gsigma_err"].to_numpy()

                    expected = power_law(x_val, *popt_mc)
                    residuals = (y_val - expected) / yerr_val
                    chi2 = np.sum(residuals**2)
                    ndf = len(x_val)
                    chi2_ndf = chi2 / ndf if ndf > 0 else np.nan

                    text_str = rf"$\chi^2/\mathrm{{NDF}} = {chi2:.2f}/{ndf} = {chi2_ndf:.2f}$"

                    # Bottom right corner text block
                    ax.text(
                        0.95,
                        0.08,
                        text_str,
                        transform=ax.transAxes,
                        fontsize=10,
                        verticalalignment="bottom",
                        horizontalalignment="right",
                        bbox=dict(
                            boxstyle="round,pad=0.3",
                            facecolor="white",
                            alpha=0.85,
                            edgecolor="none",
                        ),
                    )

            # --- Formatting ---
            if row_idx == 0:
                ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
            if row_idx == n_rows - 1:
                ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
            if plane == 0:
                ax.set_ylabel(
                    f"{row_labels[row_idx]}\n" + r"$\sigma_G$ [MeV/cm]",
                    fontsize=12,
                )

            ax.legend(fontsize=10, loc="upper left", framealpha=0.9)
            ax.grid(True, linestyle=":", alpha=0.5)

            # Enforce 1:1 square aspect ratio for every subplot
            ax.set_box_aspect(1)

    fig.tight_layout(rect=[0, 0, 1, 1])
    fig.savefig(f"{out_prefix}_data_vs_mc_fit.pdf")
    plt.show()

In [ ]:
plot_data_vs_mc_fit(
    data_results=data_results,
    mc_results=mc_results,
    mc_params=mc_params,
    particle=particle,
    out_prefix="langau_rr_data_vs_mc",
)

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np


def inspect_all_rr_slices(
    hit_dfs,
    rr_start=4,
    rr_end=40,
    rr_step=1,
    tpc=0,
    dedx_col="dedx",
    hist_nbins=150,
    hist_xmin=0.0,
    hist_xmax=10.0,
    first_stage_range=(0.5, 20.0),
    max_par_err=1.0,
):
    """Loops over 1 cm RR slices from rr_start to rr_end and plots Stage 1,

    Stage 2 window, and Stage 2 fit results across all planes.
    """
    n_planes = len(hit_dfs)
    rr_ranges = [(r, r + rr_step) for r in range(rr_start, rr_end, rr_step)]

    # Style definitions
    point_color = "#2b5c8f"
    window_color = "#8B0000"  # Dark red for the Stage 2 window

    for rr_min, rr_max in rr_ranges:
        rr_center = 0.5 * (rr_min + rr_max)

        fig = plt.figure(figsize=(8.5 * n_planes, 7.5))
        gs = gridspec.GridSpec(1, n_planes, wspace=0.25)

        x_eval = np.linspace(hist_xmin, hist_xmax, 400)
        max_y_global = 0.0
        axes = []

        for plane_idx, df in enumerate(hit_dfs):
            ax = fig.add_subplot(gs[0, plane_idx])
            axes.append(ax)

            if df is None:
                ax.set_title(
                    f"Plane {plane_idx}: No Data", fontsize=17, pad=10
                )
                continue

            sub = df if tpc == -1 else df[df["tpc"] == tpc]
            sub = sub[(sub["rr"] >= rr_min) & (sub["rr"] < rr_max)]
            if len(sub) < 10:
                ax.set_title(
                    f"Plane {plane_idx}: Low Stats ({len(sub)} hits)",
                    fontsize=17,
                    pad=10,
                )
                continue

            # Build ROOT histogram from selection
            hname = f"h_p{plane_idx}_rr{rr_center:.1f}"
            hist = th1_from_series(
                sub[dedx_col], hname, "", hist_nbins, hist_xmin, hist_xmax
            )

            bin_edges = np.linspace(hist_xmin, hist_xmax, hist_nbins + 1)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            counts = np.array(
                [hist.GetBinContent(i) for i in range(1, hist_nbins + 1)]
            )
            errors = np.array(
                [hist.GetBinError(i) for i in range(1, hist_nbins + 1)]
            )

            if len(counts) > 0:
                max_y_global = max(max_y_global, np.max(counts + errors))

            # 1. Plot Data (LaTeX formatted N_{hits})
            data_line = ax.errorbar(
                bin_centers,
                counts,
                yerr=errors,
                fmt="o",
                color=point_color,
                ecolor=point_color,
                elinewidth=1.0,
                alpha=0.6,
                markersize=3.5,
                capsize=1.5,
                label=rf"$N_{{\text{{hits}}}}$ = {len(sub)}",
            )

            # 2. Execute Fits
            fit_res = langau_fit_two_stage(
                hist,
                first_stage_range=first_stage_range,
                max_par_err=max_par_err,
            )

            s1_line = None
            s2_window_line = None
            s2_line = None

            if fit_res is not None:
                # --- Draw Stage 1 Fit ---
                f1 = fit_res.get("stage1_func")
                p1 = fit_res.get("stage1_pars")
                e1 = fit_res.get("stage1_errs")

                if f1 is not None and p1 is not None:
                    y_s1 = [f1.Eval(x) for x in x_eval]

                    mpv1_val, mpv1_err = p1[1], (e1[1] if e1 is not None else 0.0)
                    gsigma1_val = (
                        p1[3]
                        if len(p1) > 3
                        else (p1[2] if len(p1) > 2 else 0.0)
                    )
                    gsigma1_err = (
                        e1[3]
                        if (e1 is not None and len(e1) > 3)
                        else (e1[2] if (e1 is not None and len(e1) > 2) else 0.0)
                    )

                    label_s1 = (
                        f"Stage 1 Fit:\n"
                        f"  MPV = {mpv1_val:.2f} ± {mpv1_err:.2f}\n"
                        f"  $\sigma_G$ = {gsigma1_val:.2f} ± {gsigma1_err:.2f}"
                    )

                    (s1_line,) = ax.plot(
                        x_eval,
                        y_s1,
                        color="black",
                        linestyle="--",
                        linewidth=2.2,
                        alpha=0.85,
                        label=label_s1,
                    )

                # --- Draw Stage 2 Window Limits (Dark Red) ---
                s2_range = fit_res.get("stage2_range")
                if s2_range is not None:
                    s2_window_line = ax.axvline(
                        s2_range[0],
                        color=window_color,
                        linestyle=":",
                        linewidth=2.4,
                        label=f"S2 Window [{s2_range[0]:.2f}, {s2_range[1]:.2f}]",
                    )
                    ax.axvline(
                        s2_range[1],
                        color=window_color,
                        linestyle=":",
                        linewidth=2.4,
                    )

                # --- Draw Stage 2 Fit ---
                func = fit_res["func"]
                p2 = fit_res["pars"]
                e2 = fit_res["errs"]
                stage_used = fit_res.get("fit_stage", 2)

                mpv2_val, mpv2_err = p2[1], e2[1]
                gsigma2_val = (
                    p2[3] if len(p2) > 3 else (p2[2] if len(p2) > 2 else 0.0)
                )
                gsigma2_err = (
                    e2[3] if len(e2) > 3 else (e2[2] if len(e2) > 2 else 0.0)
                )

                fit_color = "tab:green" if stage_used == 2 else "purple"
                fit_style = "-" if stage_used == 2 else "-."
                stage_label = f"Stage {stage_used}"

                label_s2 = (
                    f"{stage_label} Fit:\n"
                    f"  MPV = {mpv2_val:.2f} ± {mpv2_err:.2f}\n"
                    f"  $\sigma_G$ = {gsigma2_val:.2f} ± {gsigma2_err:.2f}"
                )

                y_fit = [func.Eval(x) for x in x_eval]
                (s2_line,) = ax.plot(
                    x_eval,
                    y_fit,
                    color=fit_color,
                    linestyle=fit_style,
                    linewidth=2.6,
                    label=label_s2,
                )

            ax.set_title(f"Plane {plane_idx}", fontsize=17, pad=10)
            ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=16)
            ax.tick_params(axis="both", which="major", labelsize=12)
            ax.grid(True, linestyle=":", alpha=0.5)

            # Explicit re-ordering of legend handles and labels
            handles_raw = [s1_line, s2_line, s2_window_line, data_line]
            ordered_handles = [h for h in handles_raw if h is not None]
            ordered_labels = [h.get_label() for h in ordered_handles]

            ax.legend(
                ordered_handles,
                ordered_labels,
                loc="upper right",
                fontsize=14,
                framealpha=0.9,
                labelspacing=0.7,
                handlelength=2.0,
            )

            if plane_idx > 0:
                ax.set_ylabel("")
            else:
                ax.set_ylabel("Hits / Bin", fontsize=16)

            ax.set_box_aspect(1)

        # Synchronize Y-limits
        if max_y_global > 0:
            for ax in axes:
                ax.set_ylim(0, max_y_global * 1.35)

        tpc_str = f"TPC {tpc}" if tpc != -1 else "TPC Combined"
        fig.suptitle(
            rf"Langau Fits | {tpc_str} | Range: {rr_min} $\leq$ RR < {rr_max} cm",
            fontsize=19,
            y=1.02,
        )
        plt.tight_layout()
        plt.show()

In [ ]:
# Inputs
mc_hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
data_hit_dfs = [data_hit0_df, data_hit1_df, data_hit2_df]

# --- Run for MC (4 to 40 cm in 1 cm steps) ---
inspect_all_rr_slices(
    hit_dfs=data_hit_dfs,rr_start=4, rr_end=40, tpc=0
)
